In [1]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# lerobot
from lerobot.record import record, RecordConfig, DatasetRecordConfig
from lerobot.constants import OBS_ENV_STATE 

# torch
from torch import cuda

# gemini
from google import genai

# utils
import pprint
import json
from PIL import Image
from dotenv import load_dotenv
import IPython.display as display

# my code
from robot.robot_config import robot_config
from robot.robot_const import FOLDED_START_POSE
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR, DATASETS_DIR
from src.utils import check_resume
from gemini.gemini_lerobot_processor import GeminiAnnotateProcessorStep
from gemini.gemini_prompts import YELLOW_CAR_PROMPT


# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


#### 1. Set Evaluation Params:

In [2]:
PUSH_TO_HUB              = False
OVERRIDE_POLICY_FEATURES = True                        # used when I need to remove visual features from the policy
MANUAL_ANNOTATION        = False                       # used when manual annotation overrides is used - e.g. when using a pre-made dataset
SAVE_TO_DATASET          = True
REPO_NAME                = 'so101_car_pick_and_place'
EXPERIMENT_NAME          = 'extended_v5'
POLICY_CHECKPOINT        = '060000'
POLICY_TYPE              = 'act'
NUM_EPISODES             = 5
FPS                      = 30
EPISODE_TIME_SEC         = 60
RESET_TIME_SEC           = 5
EVAL_TYPE                = '12_test_case'

Log to HF

In [3]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")

#### 2. Initialize Policy and Overrides:

In [4]:
assert len(POLICY_CHECKPOINT) == 6

# Resolve path or HF id
if POLICY_CHECKPOINT is None:
    POLICY_CHECKPOINT = "last"
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    print('Loading policy from HF')
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{REPO_NAME}-{EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore
pprint.pprint(policy_config.input_features)

{'observation.environment_state': PolicyFeature(type=<FeatureType.ENV: 'ENV'>,
                                                shape=(3,)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                             shape=(3, 480, 640)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                               shape=(3, 480, 640)),
 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                    shape=(6,))}


Policy Overrides:

In [5]:
print(f"Original n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")
# policy_config.n_action_steps = 100
# policy_config.temporal_ensemble_coeff = 0.01
print(f"Override n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")

Original n_action_steps = 100, temporal_ensemble = None
Override n_action_steps = 100, temporal_ensemble = None


#### 3. Specify additional feature for Gemini annotation

In [6]:
new_feature_name = OBS_ENV_STATE
new_feature_meta = {
    "dtype": "float32",
    "names": [
        "x_px",
        "y_px",
        "rotation_deg"
    ],
    "shape":(3,)
}
new_feature_list = [{new_feature_name: new_feature_meta}]

And build the preprocessor that will supply this data during inference:

In [7]:
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))
gemini_processor = GeminiAnnotateProcessorStep(client = client, prompt = YELLOW_CAR_PROMPT, debug = False)

Override policy inputs - if needed:

In [8]:
if OVERRIDE_POLICY_FEATURES:
    override_input_features = {
        "observation.state": {
            "type" : "STATE",
            "shape": [6]
        },
        "observation.environment_state": {
            "type" : "STATE",
            "shape": [3]
        },
    }
    policy_config.input_features = override_input_features

#### **For Testing only - manual override**

In [9]:
MANUAL_ANNOTATION_DIR = DATASETS_DIR / REPO_NAME / 'annotation'
ANNOTATION_PATH = MANUAL_ANNOTATION_DIR /'annotations.json'
EP = 0

if MANUAL_ANNOTATION:
    with open(ANNOTATION_PATH, 'r') as f:
        data = json.load(f)
        episode = data[EP]
    annotation = (episode['x_px'], episode['y_px'], episode['theta_deg'])
    print(f"x_px: {annotation[0]}, y_px: {annotation[1]}, theta_deg: {annotation[2]}")
    img = Image.open(MANUAL_ANNOTATION_DIR / f'ep_{EP:03d}_annotation.jpg').resize((640 // 2, 480 // 2))
    display.display(img)
    gemini_processor = GeminiAnnotateProcessorStep(manual_annotation = annotation)

**End of testing block**

#### 4. Build Evaluation Dataset
Used for data storage of the evaluation dataset

In [10]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}-{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

#### 5. Run inference while calling Gemini for environemnt evaluation

In [ ]:
check_resume(dataset_path)
record(rc, 
    extra_robot_processor = [gemini_processor],
    extra_features        = new_feature_list,
    save_to_ds            = SAVE_TO_DATASET,
    reset_pose            = FOLDED_START_POSE,
    give_feedback         = False,
    log_to_file           = False)

INFO 2025-11-22 10:55:08 t/record.py:434 {'dataset': {'episode_time_s': 60,
             'fps': 30,
             'num_episodes': 5,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': True,
             'push_to_hub': False,
             'rename_map': {},
             'repo_id': 'jonathm126/eval_so101_car_pick_and_place_extended_v5-12_test_case',
             'reset_time_s': 5,
             'root': '/home/jonathan/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_car_pick_and_place/extended_v5-12_test_case',
             'single_task': 'eval dataset for so101_car_pick_and_place with '
                            'policy extended_v5, mode = 12_test_case',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 100,
            'device': 'cuda',
            'dim_feedforward': 3200,
            'dim_model'

Loading weights from local directory


INFO 2025-11-22 10:55:11 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2025-11-22 10:55:13 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2025-11-22 10:55:13 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-11-22 10:55:14 ls/utils.py:227 Recording episode 1
[2025-11-22T08:55:15Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2025-11-22T08:55:15Z WARN  wgpu_hal::gles::egl] No config found!
[2025-11-22T08:55:15Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
INFO 2025-11-22 10:55:15 /models.py:6490 AFC is enabled with max remote calls: 10.
[2025-11-22T08:55:15Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2025-11-22T08:55:15Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-11-22T08:55:15Z WARN  w

Waiting for image writer to terminate...
Value(False)
Value(False)


INFO 2025-11-22 10:56:20 /models.py:6490 AFC is enabled with max remote calls: 10.
INFO 2025-11-22 10:56:25 _client.py:1025 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-robotics-er-1.5-preview:generateContent "HTTP/1.1 200 OK"


RuntimeError: Failed in Client call with 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}

Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arr